In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from datetime import date

fake = Faker()
np.random.seed(42)

def generate_cardholder_data(n=1000):
    data = []
    for _ in range(n):
        p = fake.profile()
        balance = np.random.randint(500, 25000)
        age = date.today().year - p['birthdate'].year
        
        # Synthetic Logic: Delinquency based on high balance + low age
        prob = (balance / 25000) * 0.5 + (1 - (age / 80)) * 0.3 + np.random.random() * 0.2
        is_delinquent = 1 if prob > 0.8 else 0
        
        data.append({
            'Name': p['name'], 
            'Sex': p['sex'], 
            'Age': age,
            'Occupation': p['job'], # FIXED: changed from 'occupation' to 'job'
            'Location': p['residence'].split('\n')[-1],
            'Balance': balance, 
            'Is_Delinquent': is_delinquent
        })
    return pd.DataFrame(data)

# 1. Generate and Prepare Data
df = generate_cardholder_data()
le = LabelEncoder()
df['Sex_Code'] = le.fit_transform(df['Sex'])

# 2. Train Logistic Regression
X = df[['Balance', 'Age', 'Sex_Code']]
y = df['Is_Delinquent']
model = LogisticRegression()
model.fit(X, y)

# 3. Identify Anomalies
# We look for high "Residuals" (where actual status != predicted probability)
df['Prob_Delinquent'] = model.predict_proba(X)[:, 1]
# Anomaly = Actual Delinquent but very low predicted probability, OR 
#           Actual Current but very high predicted probability.
df['Anomaly_Score'] = abs(df['Is_Delinquent'] - df['Prob_Delinquent'])

# Filter for the top 5% of anomalies
anomalies = df[df['Anomaly_Score'] > df['Anomaly_Score'].quantile(0.95)]

print("### DATA ANOMALY REPORT")
print("The following accounts defy the standard risk model profile:\n")
for idx, row in anomalies.head(10).iterrows():
    status = "Delinquent" if row['Is_Delinquent'] == 1 else "Current"
    print(f"* {row['Name']} ({row['Occupation']})")
    print(f"  - Actual Status: {status} | Predicted Risk: {row['Prob_Delinquent']:.2%}")
    print(f"  - Balance: ${row['Balance']:,} | Age: {row['Age']}")
    print("-" * 30)

### DATA ANOMALY REPORT
The following accounts defy the standard risk model profile:

* Tonya Petty (Adult nurse)
  - Actual Status: Delinquent | Predicted Risk: 49.00%
  - Balance: $24,154 | Age: 24
------------------------------
* Michael Pineda DVM (Control and instrumentation engineer)
  - Actual Status: Current | Predicted Risk: 89.27%
  - Balance: $24,397 | Age: 7
------------------------------
* Robert Hale (Multimedia specialist)
  - Actual Status: Delinquent | Predicted Risk: 38.29%
  - Balance: $20,724 | Age: 11
------------------------------
* Antonio Anderson (Community arts worker)
  - Actual Status: Delinquent | Predicted Risk: 43.23%
  - Balance: $23,776 | Age: 22
------------------------------
* Susan Serrano (Engineer, mining)
  - Actual Status: Delinquent | Predicted Risk: 14.30%
  - Balance: $20,041 | Age: 20
------------------------------
* Sara Reed (Administrator, education)
  - Actual Status: Current | Predicted Risk: 31.23%
  - Balance: $24,276 | Age: 30
-------